# 🐝 Cahier de Test — Saison Apicole Complète

Simulation d'une saison apicole via l'API REST de **Rucher Manager**.

- **10 membres** (admin + 9 créés)
- **3 ruchers** 
- **15 ruches associatives** + **5 ruches privatives**
- Visites de mars à septembre
- Récoltes de miel avec catégories et ownership
- Mise en pot et ventes
- Trésorerie et inventaire
- Bilan final

In [ ]:
import requests, random, json
from datetime import datetime, timedelta

BASE = "http://127.0.0.1:7080/api"
random.seed(42)

# Login admin
r = requests.post(f"{BASE}/users/login", json={"email": "admin@rucher.local", "password": "admin1234"})
assert r.status_code == 200, f"Login failed: {r.text}"
TOKEN = r.json()["access_token"]
H = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

print("✅ Connecté en admin")
print(f"   Token: {TOKEN[:30]}...")

## 1. Création des 10 membres

In [ ]:
MEMBRES = [
    # admin@rucher.local déjà créé (id=1)
    {"name": "Marie Dupont",    "email": "marie@rucher.local",    "roles": ["yard_manager"]},
    {"name": "Jean Martin",     "email": "jean@rucher.local",     "roles": ["yard_manager"]},
    {"name": "Sophie Bernard",  "email": "sophie@rucher.local",   "roles": ["treasurer"]},
    {"name": "Pierre Moreau",   "email": "pierre@rucher.local",   "roles": ["yard_manager"]},
    {"name": "Lucie Petit",     "email": "lucie@rucher.local",    "roles": ["member"]},
    {"name": "Antoine Leroy",   "email": "antoine@rucher.local",  "roles": ["member"]},
    {"name": "Claire Roux",     "email": "claire@rucher.local",   "roles": ["yard_manager"]},
    {"name": "Thomas Garcia",   "email": "thomas@rucher.local",   "roles": ["member"]},
    {"name": "Émilie Fournier", "email": "emilie@rucher.local",   "roles": ["member"]},
]

user_ids = [1]  # admin
for m in MEMBRES:
    r = requests.post(f"{BASE}/users/", json={
        "name": m["name"], "email": m["email"],
        "password": "test1234", "roles": m["roles"]
    }, headers=H)
    assert r.status_code == 201, f"Erreur création {m['name']}: {r.text}"
    uid = r.json()["id"]
    user_ids.append(uid)
    print(f"  👤 {m['name']} (id={uid}, rôles={m['roles']})")

print(f"\n✅ {len(user_ids)} membres créés (dont admin)")
# yard_managers ids pour les visites et récoltes
yard_manager_ids = [user_ids[0], user_ids[1], user_ids[2], user_ids[4], user_ids[7]]
print(f"   Responsables rucher: {yard_manager_ids}")

## 2. Création des 3 ruchers

In [ ]:
RUCHERS = [
    {"name": "Les Tilleuls",   "location": "Chemin des Abeilles, 69001 Lyon",   "latitude": 45.764, "longitude": 4.835},
    {"name": "Les Acacias",    "location": "Route du Miel, 69003 Lyon",         "latitude": 45.753, "longitude": 4.858},
    {"name": "Le Châtaignier", "location": "Lieu-dit La Ruche, 69007 Lyon",     "latitude": 45.734, "longitude": 4.841},
]

apiary_ids = []
for a in RUCHERS:
    r = requests.post(f"{BASE}/apiaries/", json=a, headers=H)
    assert r.status_code == 201, f"Erreur rucher: {r.text}"
    aid = r.json()["id"]
    apiary_ids.append(aid)
    print(f"  🏕️ {a['name']} (id={aid})")

print(f"\n✅ {len(apiary_ids)} ruchers créés")

## 3. Création des 20 ruches (15 asso + 5 privées)

In [ ]:
hive_ids_asso = []
hive_ids_priv = []

# 15 ruches associatives réparties sur les 3 ruchers (5 par rucher)
for i in range(15):
    apiary_id = apiary_ids[i // 5]
    name = f"Asso-{i+1:02d}"
    napi = f"FR-69-{1000+i}"
    mgr = [random.choice(yard_manager_ids)]
    r = requests.post(f"{BASE}/apiaries/{apiary_id}/hives", json={
        "name": name, "napi_number": napi, "ownership": "associative",
        "status": "active", "notes": f"Ruche associative #{i+1}",
        "manager_ids": mgr
    }, headers=H)
    assert r.status_code == 201, f"Erreur ruche asso: {r.text}"
    hid = r.json()["id"]
    hive_ids_asso.append(hid)
    print(f"  🐝 {name} → rucher {apiary_id} (id={hid}, asso)")

# 5 ruches privées (1 par rucher Tilleuls=2, Acacias=2, Châtaignier=1)
priv_owners = [user_ids[1], user_ids[2], user_ids[4], user_ids[5], user_ids[7]]
priv_apiaries = [apiary_ids[0], apiary_ids[0], apiary_ids[1], apiary_ids[1], apiary_ids[2]]
for i in range(5):
    name = f"Priv-{i+1:02d}"
    napi = f"FR-69-{2000+i}"
    r = requests.post(f"{BASE}/apiaries/{priv_apiaries[i]}/hives", json={
        "name": name, "napi_number": napi, "ownership": "private",
        "status": "active", "notes": f"Ruche privée de {MEMBRES[priv_owners.index(priv_owners[i]) if priv_owners[i] != 1 else 0]['name'] if priv_owners[i] != 1 else 'admin'}",
        "manager_ids": [priv_owners[i]]
    }, headers=H)
    assert r.status_code == 201, f"Erreur ruche privée: {r.text}"
    hid = r.json()["id"]
    hive_ids_priv.append(hid)
    print(f"  🏠 {name} → rucher {priv_apiaries[i]} (id={hid}, privée)")

all_hive_ids = hive_ids_asso + hive_ids_priv
print(f"\n✅ {len(hive_ids_asso)} ruches associatives + {len(hive_ids_priv)} ruches privées = {len(all_hive_ids)} ruches")

## 4. Simulation des visites (mars → septembre 2026)

In [ ]:
# Visites toutes les 2 semaines de mars à septembre sur un échantillon de ruches
visit_count = 0
months_visits = {m: 0 for m in range(3, 10)}  # mars=3 à sept=9

for month in range(3, 10):
    for fortnight in [1, 15]:
        visit_date = datetime(2026, month, fortnight, 10, 0).isoformat()
        # Chaque visite couvre ~8 ruches aléatoires
        hives_to_visit = random.sample(all_hive_ids, min(8, len(all_hive_ids)))
        for hive_id in hives_to_visit:
            # Scores saisonniers réalistes
            if month <= 4:
                brood = random.randint(3, 6)
                reserves = random.randint(3, 5)
            elif month <= 7:
                brood = random.randint(6, 9)
                reserves = random.randint(5, 8)
            else:
                brood = random.randint(4, 7)
                reserves = random.randint(6, 9)

            supers_delta = 0
            supers_count = 0
            if month >= 5 and month <= 7:
                supers_delta = random.choice([0, 0, 1, 1, 2])
                supers_count = random.randint(1, 3)

            visit_data = {
                "hive_id": hive_id,
                "visit_date": visit_date,
                "queen_seen": random.random() > 0.3,
                "brood_score": brood,
                "reserves_score": reserves,
                "supers_delta": supers_delta,
                "supers_count": supers_count,
                "notes": random.choice([
                    "Colonie dynamique", "Reine vue, ponte régulière",
                    "Réserves correctes", "Ajout hausse nécessaire",
                    "Quelques cellules royales", "Traitement varroa prévu",
                    "Belle activité au trou de vol", "Colonie calme",
                ]),
            }
            r = requests.post(f"{BASE}/visits/", json=visit_data, headers=H)
            if r.status_code == 201:
                visit_count += 1
                months_visits[month] += 1

print(f"✅ {visit_count} visites créées")
for m, c in months_visits.items():
    mois = ["", "", "", "Mars", "Avril", "Mai", "Juin", "Juillet", "Août", "Septembre"][m]
    print(f"   {mois}: {c} visites")

## 5. Catégories de miel & Récoltes

In [ ]:
# Créer les catégories de miel
CATEGORIES = ["Acacia", "Toutes fleurs", "Châtaignier", "Lavande"]
cat_ids = []
for name in CATEGORIES:
    r = requests.post(f"{BASE}/honey/categories", json={"name": name}, headers=H)
    assert r.status_code == 201, f"Erreur catégorie: {r.text}"
    cat_ids.append(r.json()["id"])
    print(f"  🍯 Catégorie: {name} (id={r.json()['id']})")

# Récoltes — printemps (juin) et été (juillet-août)
harvest_ids = []
total_kg_asso = 0
total_kg_priv = 0

RECOLTES = [
    # Récolte printemps — Acacia (rucher Tilleuls, asso)
    {"apiary_id": apiary_ids[0], "category_id": cat_ids[0], "ownership": "associative",
     "quantity_kg": 45.5, "nb_supers": 5, "nb_frames": 30,
     "harvest_date": "2026-06-10T09:00:00", "notes": "Récolte acacia printemps, très bon cru"},
    # Récolte printemps — Toutes fleurs (rucher Acacias, asso)
    {"apiary_id": apiary_ids[1], "category_id": cat_ids[1], "ownership": "associative",
     "quantity_kg": 38.0, "nb_supers": 4, "nb_frames": 24,
     "harvest_date": "2026-06-15T09:00:00", "notes": "Toutes fleurs, bon rendement"},
    # Récolte été — Châtaignier (rucher Châtaignier, asso)
    {"apiary_id": apiary_ids[2], "category_id": cat_ids[2], "ownership": "associative",
     "quantity_kg": 52.0, "nb_supers": 6, "nb_frames": 36,
     "harvest_date": "2026-07-12T09:00:00", "notes": "Châtaignier corsé, belle couleur ambrée"},
    # Récolte été — Toutes fleurs (rucher Tilleuls, asso)
    {"apiary_id": apiary_ids[0], "category_id": cat_ids[1], "ownership": "associative",
     "quantity_kg": 35.0, "nb_supers": 4, "nb_frames": 22,
     "harvest_date": "2026-07-25T09:00:00", "notes": "Deuxième récolte toutes fleurs"},
    # Récolte été — Lavande (rucher Acacias, asso)
    {"apiary_id": apiary_ids[1], "category_id": cat_ids[3], "ownership": "associative",
     "quantity_kg": 28.5, "nb_supers": 3, "nb_frames": 18,
     "harvest_date": "2026-08-05T09:00:00", "notes": "Lavande, quantité modeste mais qualité top"},
    # Récoltes privées
    {"apiary_id": apiary_ids[0], "category_id": cat_ids[0], "ownership": "private",
     "quantity_kg": 12.0, "nb_supers": 2, "nb_frames": 8,
     "harvest_date": "2026-06-12T09:00:00", "notes": "Récolte privée Marie — acacia"},
    {"apiary_id": apiary_ids[1], "category_id": cat_ids[1], "ownership": "private",
     "quantity_kg": 15.5, "nb_supers": 2, "nb_frames": 10,
     "harvest_date": "2026-07-14T09:00:00", "notes": "Récolte privée Jean — toutes fleurs"},
    {"apiary_id": apiary_ids[2], "category_id": cat_ids[2], "ownership": "private",
     "quantity_kg": 8.0, "nb_supers": 1, "nb_frames": 6,
     "harvest_date": "2026-08-01T09:00:00", "notes": "Récolte privée Pierre — châtaignier"},
]

for rec in RECOLTES:
    r = requests.post(f"{BASE}/honey/", json=rec, headers=H)
    assert r.status_code == 201, f"Erreur récolte: {r.text}"
    hid = r.json()["id"]
    harvest_ids.append(hid)
    kg = rec["quantity_kg"]
    if rec["ownership"] == "associative":
        total_kg_asso += kg
    else:
        total_kg_priv += kg
    emoji = "🏛️" if rec["ownership"] == "associative" else "🏠"
    print(f"  {emoji} Récolte #{hid}: {kg}kg — {rec['notes'][:40]}")

print(f"\n✅ {len(harvest_ids)} récoltes créées")
print(f"   🏛️ Associatif: {total_kg_asso} kg")
print(f"   🏠 Privé: {total_kg_priv} kg")
print(f"   📊 Total: {total_kg_asso + total_kg_priv} kg")

## 6. Mise en pot

In [ ]:
# Mise en pot : pots 500g et 250g pour les récoltes asso, quelques pots privés
jar_ids = []

POTS = [
    # Récolte asso #1 (45.5kg acacia) → 60 pots 500g + 40 pots 250g
    {"harvest_id": harvest_ids[0], "ownership": "associative", "jar_weight_g": 500, "quantity": 60, "unit_price": 8.0},
    {"harvest_id": harvest_ids[0], "ownership": "associative", "jar_weight_g": 250, "quantity": 40, "unit_price": 5.0},
    # Récolte asso #2 (38kg toutes fleurs) → 50 pots 500g
    {"harvest_id": harvest_ids[1], "ownership": "associative", "jar_weight_g": 500, "quantity": 50, "unit_price": 7.5},
    # Récolte asso #3 (52kg châtaignier) → 70 pots 500g + 30 pots 250g
    {"harvest_id": harvest_ids[2], "ownership": "associative", "jar_weight_g": 500, "quantity": 70, "unit_price": 9.0},
    {"harvest_id": harvest_ids[2], "ownership": "associative", "jar_weight_g": 250, "quantity": 30, "unit_price": 5.5},
    # Récolte asso #4 (35kg toutes fleurs) → 45 pots 500g
    {"harvest_id": harvest_ids[3], "ownership": "associative", "jar_weight_g": 500, "quantity": 45, "unit_price": 7.5},
    # Récolte asso #5 (28.5kg lavande) → 35 pots 500g + 20 pots 250g
    {"harvest_id": harvest_ids[4], "ownership": "associative", "jar_weight_g": 500, "quantity": 35, "unit_price": 10.0},
    {"harvest_id": harvest_ids[4], "ownership": "associative", "jar_weight_g": 250, "quantity": 20, "unit_price": 6.0},
    # Récoltes privées
    {"harvest_id": harvest_ids[5], "ownership": "private", "jar_weight_g": 500, "quantity": 15, "unit_price": 8.0},
    {"harvest_id": harvest_ids[6], "ownership": "private", "jar_weight_g": 500, "quantity": 20, "unit_price": 7.5},
    {"harvest_id": harvest_ids[7], "ownership": "private", "jar_weight_g": 250, "quantity": 20, "unit_price": 5.0},
]

for pot in POTS:
    r = requests.post(f"{BASE}/honey/jars", json=pot, headers=H)
    assert r.status_code == 201, f"Erreur pot: {r.text}"
    jid = r.json()["id"]
    jar_ids.append(jid)
    emoji = "🏛️" if pot["ownership"] == "associative" else "🏠"
    print(f"  🫙 {emoji} {pot['quantity']}x {pot['jar_weight_g']}g @ {pot['unit_price']}€ (jar_id={jid})")

print(f"\n✅ {len(jar_ids)} lots de pots créés")

# Vérifier le stock
r = requests.get(f"{BASE}/honey/jars/stock", headers=H)
print("\n📦 Stock de pots:")
for s in r.json():
    emoji = "🏛️" if s["ownership"] == "associative" else "🏠"
    print(f"   {emoji} {s['jar_weight_g']}g: {s['stock']} en stock (initial: {s['initial']})")

## 7. Ventes de miel (avec auto-compta)

In [ ]:
# Simuler 8 ventes
VENTES = [
    {"jar_id": jar_ids[0], "quantity": 10, "buyer": "Marché de Vaise"},
    {"jar_id": jar_ids[0], "quantity": 5,  "buyer": "Mme Leblanc"},
    {"jar_id": jar_ids[1], "quantity": 8,  "buyer": "Fête du miel"},
    {"jar_id": jar_ids[2], "quantity": 12, "buyer": "Épicerie Bio du Coin"},
    {"jar_id": jar_ids[3], "quantity": 15, "buyer": "Marché Croix-Rousse"},
    {"jar_id": jar_ids[6], "quantity": 6,  "buyer": "M. Durand — lavande"},
    {"jar_id": jar_ids[8], "quantity": 5,  "buyer": "Vente privée Marie"},       # privé
    {"jar_id": jar_ids[9], "quantity": 8,  "buyer": "Amis de Jean"},             # privé
]

total_ventes = 0
for v in VENTES:
    r = requests.post(f"{BASE}/honey/sales", json=v, headers=H)
    assert r.status_code == 201, f"Erreur vente: {r.text}"
    sale = r.json()
    total_ventes += sale["total_amount"]
    print(f"  💰 Vente #{sale['id']}: {v['quantity']}x → {sale['total_amount']:.2f}€ — {v['buyer']}")

print(f"\n✅ {len(VENTES)} ventes réalisées")
print(f"   💰 Total ventes: {total_ventes:.2f} €")

## 8. Trésorerie (cotisations, achats, ventes)

In [ ]:
# Transactions de trésorerie (les ventes miel sont déjà auto-créées)
TRANSACTIONS = [
    # Cotisations des 10 membres
    *[{"transaction_type": "income", "category": "membership", "amount": 30.0,
       "description": f"Cotisation 2026 — {MEMBRES[i-1]['name'] if i > 0 else 'Admin'}",
       "date": f"2026-01-{10+i}T10:00:00"} for i in range(10)],
    # Achats matériel
    {"transaction_type": "expense", "category": "material", "amount": 250.0,
     "description": "10 cadres de corps Dadant", "supplier": "Apidistri", "date": "2026-03-05T10:00:00"},
    {"transaction_type": "expense", "category": "material", "amount": 180.0,
     "description": "5 hausses Dadant complètes", "supplier": "Thomas Apiculture", "date": "2026-04-10T10:00:00"},
    {"transaction_type": "expense", "category": "material", "amount": 95.0,
     "description": "Enfumoir + lève-cadre", "supplier": "Apidistri", "date": "2026-03-15T10:00:00"},
    {"transaction_type": "expense", "category": "material", "amount": 320.0,
     "description": "Extracteur 4 cadres tangentiel", "supplier": "Lega France", "date": "2026-05-20T10:00:00"},
    # Traitement varroa
    {"transaction_type": "expense", "category": "treatment", "amount": 85.0,
     "description": "Apivar — 20 lanières varroa", "supplier": "Véto-Pharma", "date": "2026-08-15T10:00:00"},
    {"transaction_type": "expense", "category": "treatment", "amount": 45.0,
     "description": "Acide oxalique 3.5%", "supplier": "Véto-Pharma", "date": "2026-09-10T10:00:00"},
]

tx_count = 0
total_in = 0
total_out = 0
for tx in TRANSACTIONS:
    r = requests.post(f"{BASE}/treasury/", json=tx, headers=H)
    assert r.status_code == 201, f"Erreur transaction: {r.text}"
    tx_count += 1
    if tx["transaction_type"] == "income":
        total_in += tx["amount"]
    else:
        total_out += tx["amount"]

print(f"✅ {tx_count} transactions créées")
print(f"   📈 Recettes (cotisations): {total_in:.2f} €")
print(f"   📉 Dépenses (matériel + traitements): {total_out:.2f} €")
print(f"   💰 + ventes miel auto-comptabilisées: {total_ventes:.2f} €")
print(f"   📊 Solde estimé: {total_in + total_ventes - total_out:.2f} €")

## 9. Inventaire (matériel apicole)

In [ ]:
INVENTAIRE = [
    {"name": "Corps Dadant 10c", "category": "Ruche", "location": "Local asso", "quantity": 8, "unit": "unité", "alert_threshold": 3},
    {"name": "Hausse Dadant", "category": "Ruche", "location": "Local asso", "quantity": 12, "unit": "unité", "alert_threshold": 5},
    {"name": "Cadre de corps ciré", "category": "Cadre", "location": "Local asso", "quantity": 50, "unit": "unité", "alert_threshold": 20},
    {"name": "Cadre de hausse ciré", "category": "Cadre", "location": "Local asso", "quantity": 40, "unit": "unité", "alert_threshold": 15},
    {"name": "Enfumoir inox", "category": "Outil", "location": "Local asso", "quantity": 3, "unit": "unité"},
    {"name": "Combinaison apiculteur", "category": "Protection", "location": "Local asso", "quantity": 6, "unit": "unité"},
    {"name": "Gants cuir", "category": "Protection", "location": "Local asso", "quantity": 10, "unit": "paire"},
    {"name": "Lève-cadre", "category": "Outil", "location": "Local asso", "quantity": 4, "unit": "unité"},
    {"name": "Extracteur tangentiel 4c", "category": "Extraction", "location": "Miellerie", "quantity": 1, "unit": "unité"},
    {"name": "Bac à désoperculer", "category": "Extraction", "location": "Miellerie", "quantity": 2, "unit": "unité"},
    {"name": "Maturateur 50kg", "category": "Extraction", "location": "Miellerie", "quantity": 2, "unit": "unité"},
    {"name": "Pot 500g verre", "category": "Conditionnement", "location": "Miellerie", "quantity": 200, "unit": "unité", "alert_threshold": 50},
    {"name": "Pot 250g verre", "category": "Conditionnement", "location": "Miellerie", "quantity": 150, "unit": "unité", "alert_threshold": 30},
    {"name": "Lanière Apivar", "category": "Traitement", "location": "Local asso", "quantity": 20, "unit": "unité"},
    {"name": "Acide oxalique 3.5%", "category": "Traitement", "location": "Local asso", "quantity": 5, "unit": "flacon"},
]

item_ids = []
for item in INVENTAIRE:
    r = requests.post(f"{BASE}/inventory/", json=item, headers=H)
    assert r.status_code == 201, f"Erreur inventaire: {r.text}"
    iid = r.json()["id"]
    item_ids.append(iid)
    print(f"  📦 {item['name']}: {item['quantity']} {item['unit']} ({item['location']})")

print(f"\n✅ {len(item_ids)} articles en inventaire")

## 10. Vérification des stats miel (ownership breakdown)

In [ ]:
# Stats globales
r = requests.get(f"{BASE}/honey/stats", headers=H)
stats = r.json()
print("📊 STATISTIQUES MIEL 2026")
print(f"   Production totale: {stats['total_kg']} kg")
print(f"   Nombre de récoltes: {stats['nb_harvests']}")

print("\n   Par type de propriété:")
for own in stats.get("by_ownership", []):
    emoji = "🏛️" if own["ownership"] == "associative" else "🏠"
    print(f"     {emoji} {own['ownership']}: {own['total_kg']} kg ({own['nb_harvests']} récoltes)")

print("\n   Par catégorie:")
for cat in stats.get("by_category", []):
    print(f"     🍯 {cat['category']}: {cat['total_kg']} kg ({cat['nb_harvests']} récoltes)")

print("\n   Par mois:")
mois_noms = {6: "Juin", 7: "Juillet", 8: "Août"}
for m in stats.get("by_month", []):
    print(f"     📅 {mois_noms.get(m['month'], m['month'])}: {m['total_kg']} kg")

# Stats filtrées asso uniquement
r2 = requests.get(f"{BASE}/honey/stats", params={"ownership": "associative"}, headers=H)
stats_asso = r2.json()
print(f"\n   🏛️ Filtre associatif seul: {stats_asso['total_kg']} kg")

# Stock de pots final
r3 = requests.get(f"{BASE}/honey/jars/stock", headers=H)
print("\n📦 STOCK DE POTS FINAL:")
for s in r3.json():
    emoji = "🏛️" if s["ownership"] == "associative" else "🏠"
    print(f"   {emoji} {s['jar_weight_g']}g: {s['stock']} restants / {s['initial']} initial (vendus: {s['sold']})")

## 11. 🏆 Bilan de fin de saison

In [ ]:
# Récapitulatif complet
r_users = requests.get(f"{BASE}/users/", headers=H)
r_apiaries = requests.get(f"{BASE}/apiaries/", headers=H)
r_treasury = requests.get(f"{BASE}/treasury/", headers=H)
r_inventory = requests.get(f"{BASE}/inventory/", headers=H)
r_sales = requests.get(f"{BASE}/honey/sales", headers=H)

users = r_users.json()
apiaries = r_apiaries.json()
treasury = r_treasury.json()
inventory = r_inventory.json()
sales_data = r_sales.json()

# Compter ruches par rucher
total_hives = 0
for a in apiaries:
    hive_count = len(a.get("hives", []))
    total_hives += hive_count

# Trésorerie
income = sum(t["amount"] for t in treasury if t["transaction_type"] == "income")
expense = sum(t["amount"] for t in treasury if t["transaction_type"] == "expense")
sales_total = sum(s["total_amount"] for s in sales_data)

print("=" * 60)
print("🐝  BILAN SAISON APICOLE 2026 — RUCHER MANAGER")
print("=" * 60)
print()
print(f"👥  Membres:                {len(users)}")
print(f"🏕️  Ruchers:                {len(apiaries)}")
for a in apiaries:
    print(f"     • {a['name']}: {len(a.get('hives', []))} ruches")
print(f"🐝  Ruches totales:         {len(all_hive_ids)}")
print(f"     • Associatives:       {len(hive_ids_asso)}")
print(f"     • Privées:            {len(hive_ids_priv)}")
print()
print(f"📋  Visites effectuées:     {visit_count}")
print()
print(f"🍯  Production miel:       {stats['total_kg']} kg")
print(f"     • Associatif:         {total_kg_asso} kg")
print(f"     • Privé:              {total_kg_priv} kg")
print(f"     • Moy./ruche (asso):  {total_kg_asso / len(hive_ids_asso):.1f} kg")
print()
print(f"🫙  Pots mis en pot:       {sum(p['quantity'] for p in POTS)}")
print(f"💰  Ventes:                {len(sales_data)} ({sales_total:.2f} €)")
print()
print(f"📈  Trésorerie:")
print(f"     Recettes:             {income:.2f} €")
print(f"     Dépenses:             {expense:.2f} €")
print(f"     Solde:                {income - expense:.2f} €")
print()
print(f"📦  Articles en inventaire: {len(inventory)}")
print()
print("=" * 60)
print("✅  Tous les tests ont réussi — la saison est complète !")
print("=" * 60)